In [68]:
from dotenv import load_dotenv
import os
from typing import TypedDict, List

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.tools import tool

from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

In [69]:
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY missing")

llm = ChatOpenAI(model="gpt-4o", temperature=0)


In [70]:
from typing import Annotated
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from typing import TypedDict, List

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

In [71]:
@tool
def search_flights(query: str) -> str:
    """Search for flights between cities."""
    return f"✈️ Flights for {query}"

@tool
def find_hotels(query: str) -> str:
    """Find hotels for a destination."""
    return f"🏨 Hotels for {query}"

@tool
def plan_itinerary(query: str) -> str:
    """Create a travel itinerary."""
    return f"🗺️ Itinerary for {query}"

tools = [search_flights, find_hotels, plan_itinerary]

llm_with_tools = llm.bind_tools(tools)


In [72]:
def agent_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])

    return {
        "messages": [response]
    }

In [73]:
tool_node = ToolNode(tools)

In [74]:
from langchain_core.messages import AIMessage

def should_continue(state: AgentState):
    last_message = state["messages"][-1]

    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return "tools"

    return "end"

In [75]:
graph = StateGraph(AgentState)

graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

graph.set_entry_point("agent")

graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "end": END
    }
)

graph.add_edge("tools", "agent")

app = graph.compile()

In [76]:
from langchain_core.messages import HumanMessage

result = app.invoke({
    "messages": [HumanMessage(content="Plan a honeymoon trip from Delhi to Bali under $3000")]
})

for msg in result["messages"]:
    print(msg)

content='Plan a honeymoon trip from Delhi to Bali under $3000' additional_kwargs={} response_metadata={} id='ec7d797e-9d2a-4375-b9c4-d4824d7b4eda'
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 97, 'total_tokens': 170, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_303cbe1edb', 'id': 'chatcmpl-DblpynEM3nDBot4k8bqd6sCKC14Tk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019df2c3-87c4-7502-8328-6b0c47fa0fa4-0' tool_calls=[{'name': 'search_flights', 'args': {'query': 'flights from Delhi to Bali'}, 'id': 'call_O5KRgjHgNVJqAYPs0rF5tVpe', 'type': 'tool_call'}, {'name': 'find_hotels', 'args': {'query': 'hotels in Bali'}, 'id': 'cal